# Phylogeny of datasets

## Minhash phylogeny - Sourmash

#### Testing

##### making sketches

In [ ]:
import sourmash
seq1 = "ATGGCA"
seq2 = "AGAGCA"

mh1 = sourmash.MinHash(n=0, ksize=3, scaled=1)
mh1.add_sequence(seq1, force=True)

mh2 = sourmash.MinHash(n=0, ksize=3, scaled=1)
mh2.add_sequence(seq2, force=True)

In [ ]:
mh1.jaccard(mh1)

### Phage Minhashing 

##### Constructing signatures using manipulations.py

In [ ]:
import os, sys 
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

from manipulations import construct_SM_sketches
construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                      k = 18, 
                      outdir = "PhageMinhash_n500_k18/", 
                      quiet = False,
                      sourmash_parameters=[500, 0])


##### iterating through kmers

In [ ]:
import sourmash
from Bio import SeqIO
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

records = list(SeqIO.parse(raw_data_path+"phagehost_KU/phage_cleaned.fasta", "fasta"))
K = 8

#only first record
print("First record:", records[0].id, len(records[0].seq))
mh1 = sourmash.MinHash(n=0, ksize=K, scaled=1)
for i in range(0, len(records[0].seq) - K + 1):
    kmer = str(records[0].seq[i:i+K])
    mh1.add_sequence(kmer, force=True)
    print(i, kmer, mh1.seq_to_hashes(kmer))

#second record
print("Second record:", records[1].id, len(records[1].seq))
mh2 = sourmash.MinHash(n=0, ksize=K, scaled=1)
for i in range(0, len(records[1].seq) - K + 1):
    kmer = str(records[1].seq[i:i+K])
    mh2.add_sequence(kmer, force=True)
    print(i, kmer, mh2.seq_to_hashes(kmer))

Comparing sketches of mh1 and mh2

In [ ]:
from tqdm import tqdm

#Constructing minhashes for all records
minhashes = []
for rec in tqdm(records, desc="Constructing minhashes for all records", unit="seq"):
    #print("Record:", rec.id, len(rec.seq))
    mh = sourmash.MinHash(n=0, ksize=K, scaled=1) #each record gets its own minhash
    for i in range(0, len(rec.seq) - K + 1):
        kmer = str(rec.seq[i:i+K])
        mh.add_sequence(kmer, force=True)
    minhashes.append(mh)

#Comparing all minhashes
similarity_matrix = dict()
for i, e in enumerate(minhashes):
    sim_inner = dict()
    for j, e2 in enumerate(minhashes):
        x = e.jaccard(minhashes[j])
        sim_inner[records[j].id.split("_")[-1]] = x
    similarity_matrix[records[i].id.split("_")[-1]] = sim_inner

### Converting all phage genomes to sketches and saving

In [ ]:
from tqdm import tqdm
from Bio import SeqIO
import sourmash, sys, os
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"
K = 12 #kmer size in nucleotides 

records = list(SeqIO.parse(raw_data_path+"phagehost_KU/phage_cleaned.fasta", "fasta"))

#Constructing minhashes for all records
minhashes = []
phage_names = []
for rec in tqdm(records, desc="Constructing minhashes for all records", unit="seq"):
    #print("Record:", rec.id, len(rec.seq))
    mh = sourmash.MinHash(n=500000, ksize=K, scaled=0) #each record gets its own minhash | scaled=1000 to limit 
    for i in range(0, len(rec.seq) - K + 1):
        kmer = str(rec.seq[i:i+K])
        mh.add_sequence(kmer, force=True)
    minhashes.append(mh)
    phage_names.append(rec.id.split("_")[-1])

if len(minhashes) != len(records):
    print("Warning: Number of minhashes does not match number of records!")
    sys.exit(1)

#Comparing all minhashes
similarity_matrix = dict()
for i, e in enumerate(minhashes):
    sim_inner = dict()
    for j, e2 in enumerate(minhashes):
        x = e.jaccard(minhashes[j])
        sim_inner[records[j].id] = x
    similarity_matrix[records[i].id] = sim_inner

#Creating output directory
if not os.path.exists(data_prod_path+f"phage_minhash_{K}/"):
    os.makedirs(data_prod_path+f"phage_minhash_{K}/")

#Saving sketches
for i in range(len(minhashes)):
    with open(data_prod_path+f"phage_minhash_{K}/{phage_names[i]}.sig", "wt") as sigfile:
        sig1 = sourmash.SourmashSignature(minhashes[i], name=phage_names[i])
        sourmash.save_signatures([sig1], sigfile)

#### Plotting similarity as heatmap

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.DataFrame(similarity_matrix)
fig = plt.figure(figsize=(10,6))
plt.title(f"MinHash Similarity of Phages {K}mer")
sns.heatmap(df, cmap="YlGnBu")

In [ ]:
selected_phages = ["Abuela", "Koroua", "Sabo", "Taid", "Ymer", "Amona", "Poppous"]

fig = plt.figure(figsize=(10,6))
plt.title(f"MinHash Similarity of Selected Phage Cluster {K}mer")
sns.heatmap(df[df.index.isin(selected_phages)][selected_phages], cmap="YlGnBu", annot=True)

### Bacteria Minhashing

##### Constructing signatures using manipulations.py

In [ ]:
import os, sys 
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

from manipulations import construct_SM_sketches
construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", 
                      k = 18, 
                      outdir = "BactMinhash_n500/", 
                      quiet = False,
                      sourmash_parameters=[500, 0])

##### [OLD WAY] iterating through kmers

Comparing sketches of mh1 and mh2

In [ ]:
from Bio import SeqIO
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

records = list(SeqIO.parse(raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", "fasta"))
K = 12

In [ ]:
from tqdm import tqdm

#Constructing minhashes for all records
minhashes = []
for rec in tqdm(records, desc="Constructing minhashes for all records", unit="seq"):
    #print("Record:", rec.id, len(rec.seq))
    mh = sourmash.MinHash(n=500000, ksize=K, scaled=0) #each record gets its own minhash | scaled=1000 to limit 
    for i in range(0, len(rec.seq) - K + 1):
        kmer = str(rec.seq[i:i+K])
        mh.add_sequence(kmer, force=True)
    minhashes.append(mh)

#Comparing all minhashes
similarity_matrix = dict()
for i, e in enumerate(minhashes):
    sim_inner = dict()
    for j, e2 in enumerate(minhashes):
        x = e.jaccard(minhashes[j])
        sim_inner[records[j].id] = x
    similarity_matrix[records[i].id] = sim_inner

In [ ]:
#Saving sketches
for i in range(len(minhashes)):
    with open(data_prod_path+f"bact_minhash_{K}/bact{i}.sig", "wt") as sigfile:
        sig1 = sourmash.SourmashSignature(minhashes[i], name=records[i].id)
        sourmash.save_signatures([sig1], sigfile)

### Working with signatures

In [ ]:
import sourmash, os, sys
K = 12
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"
minhashes = []
sig_dir = data_prod_path+"bact_minhash_37/"
print("Number of signatures", len(os.listdir(sig_dir)))

"""
### Loading sigs - individually
for file in os.listdir(sig_dir):
    try:
        single_sig = sourmash.load_one_signature(sig_dir+file)
        print(single_sig)
    except: #catching wrongful load of signatures
        print(f"Failed to load {file}, now exiting")
        sys.exit(1)
    minhashes.append(single_sig)
"""

### Loading sigs - collectively
loaded_sigs = list(sourmash.load_file_as_signatures(sig_dir))
#print(loaded_sigs[0])

Calculating the jaccard similarity

In [ ]:
from tqdm import tqdm
import numpy as np

num_sigs = len(loaded_sigs)
count_zero_jac = 0
count_nonzero_jac = 0
jac_matrix = np.zeros((num_sigs, num_sigs))

for i in tqdm(range(num_sigs), desc="Iterating through loaded sigs (Outer loop)", unit="sigs"):
    for j in range(num_sigs):
        jac = loaded_sigs[i].jaccard(loaded_sigs[j])
        if jac > 0:
            jac_matrix[i, j] = jac
            count_nonzero_jac += 1
        else:
            #no need to add at jac_matrix, as it is already filled with zeros
            count_zero_jac += 1

print(f"Jaccard sim above 0 vs at zero: {count_nonzero_jac}/{count_zero_jac}")

In [ ]:
#print([l.minhash.hashes.keys() for l in loaded_sigs])

bact_names = []
for i in range(num_sigs):
    #print(str(loaded_sigs[i]), type(loaded_sigs[i]))
    bact_names.append(str(loaded_sigs[i]))

for name in bact_names:
    print(name, type(name))

#### Plotting raw similarity as heatmap

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sim_bact_df = pd.DataFrame(jac_matrix)
sim_bact_df.columns = bact_names
sim_bact_df.index = bact_names
display(sim_bact_df)

plt.figure(figsize=(12,8))
sns.heatmap(sim_bact_df, cmap="YlGnBu")
plt.xticks([], [])
plt.yticks([], [])

#### Similarity plot ordered by family
Lookup ids in excel file

In [ ]:
hostrange_pdf = pd.read_excel(raw_data_path+"phagehost_KU/Hostrange_data_all_crisp_iso.xlsx", sheet_name="sum_hostrange", header=1)
id_lookup_bact = hostrange_pdf.set_index("Seq ID")["Species"] #fasta seq IDs + bacteria species lookup
id_lookup_bact.index = id_lookup_bact.index.str.replace("_reoriented", "", regex=False) # Adjusting index to match the format of bact_names (removing "_reoriented" suffix)
print(id_lookup_bact)

sim_bact_df = pd.DataFrame(jac_matrix)
sim_bact_df.columns = bact_names
sim_bact_df.index = bact_names
#display(sim_bact_df)

sim_bact_df = sim_bact_df.join(id_lookup_bact, how="left")
sim_bact_df["Species"] #Species is now in sim_bact_df columns
display(sim_bact_df["Species"])
sim_bact_df.loc['Species'] = list(sim_bact_df["Species"].values)+["NA"] # Add 'Species' as a row at the bottom of sim_bact_df

Sort index based on Species

In [ ]:
sim_bact_sorted_df = sim_bact_df.sort_values(by="Species", axis=0)
sim_bact_sorted_df["Species"]

In [ ]:
sim_bact_sorted_df

Sort columns based on Species

In [ ]:
display(id_lookup_bact)

Transpose to sort again

In [ ]:
sim_bact_sorted_df = sim_bact_sorted_df.T
sim_bact_sorted_df = sim_bact_sorted_df.sort_values(by="Species", axis=0)
sim_bact_sorted_df #Columns and index positions match!

#### Plotting similarity with sorted indexes

#### Plotting similarity with sorted indexes

In [ ]:
from collections import Counter

def unique_axis_labels(labels):
    counts = Counter(labels)
    l_dict = {}
    for l_uniq in set(labels):
        for i, label in enumerate(labels):
            if l_uniq == label:

                l_dict[l_uniq] = i + round(counts[l_uniq]/2)
                break
    labels_out = ["" for _ in range(len(labels))] #filled with "-" to remove duplicates
    for key, value in l_dict.items():
        labels_out[value] = key
    return labels_out


def reformat_bact_names(bact_names : list):
    new_names = []
    for name in bact_names:
        name_split = name.split(" ")
        if len(name_split) > 1: #two names
            kingdom_abbrev = name.split(" ")[0][0].strip(" ")
            new_names.append(kingdom_abbrev+". "+name.split(" ")[1])
        else:
            new_names.append(name)
    return new_names

### Prepping dataframe (without species)
sim_bact_sorted_df.to_csv(data_prod_path+"sim_bact_sorted_df.csv")
sim_bact_sorted_data_df = sim_bact_sorted_df.drop(index="Species").drop(columns=["Species"])
#display(sim_bact_sorted_data_df)
sim_bact_sorted_data_df = sim_bact_sorted_data_df.astype(float) 

### Making unique labels
print(sim_bact_sorted_df["Species"].values)
labels_abbrev = reformat_bact_names(sim_bact_sorted_df["Species"].values)
labels_unique = unique_axis_labels(labels_abbrev)

plt.figure(figsize=(12,8))
sns.heatmap(sim_bact_sorted_data_df, cmap="YlGnBu")

plt.xticks(
    ticks=range(len(sim_bact_sorted_data_df.index)+1),
    labels=labels_unique,
    rotation=90,
    fontsize=6
    )

plt.yticks(
    ticks=range(len(sim_bact_sorted_data_df.index)+1),
    labels=labels_unique,
    fontsize=6
    )

# Adding y lines for species 
y_labels = sim_bact_sorted_df["Species"].values # Get the species labels for the y-axis
species_change_indices_y = [i for i in range(1, len(y_labels)) if y_labels[i] != y_labels[i-1]] # Find the indices where the species label changes
for idy in species_change_indices_y: # Plot horizontal lines at these indices
    plt.hlines(idy-1, xmin=0, xmax=len(sim_bact_sorted_data_df.columns), colors='grey', linestyles='dashed', linewidth=0.5)

# Adding x lines for species 
x_labels = sim_bact_sorted_df["Species"].values # Get the species labels for the y-axis
species_change_indices_x = [i for i in range(1, len(x_labels)) if x_labels[i] != x_labels[i-1]] # Find the indices where the species label changes
for idx in species_change_indices_x: # Plot horizontal lines at these indices
    plt.vlines(idx-1, ymin=0, ymax=len(sim_bact_sorted_data_df.columns), colors='grey', linestyles='dashed', linewidth=0.5)

plt.title(f"Similarity of bacterial strains by family with {K}mer")

In [ ]:
from collections import Counter

def unique_axis_labels(labels):
    counts = Counter(labels)
    l_dict = {}
    for l_uniq in set(labels):
        for i, label in enumerate(labels):
            if l_uniq == label:

                l_dict[l_uniq] = i + round(counts[l_uniq]/2)
                break
    labels_out = ["" for _ in range(len(labels))] #filled with "-" to remove duplicates
    for key, value in l_dict.items():
        labels_out[value] = key
    return labels_out


def reformat_bact_names(bact_names : list):
    new_names = []
    for name in bact_names:
        name_split = name.split(" ")
        if len(name_split) > 1: #two names
            kingdom_abbrev = name.split(" ")[0][0].strip(" ")
            new_names.append(kingdom_abbrev+". "+name.split(" ")[1])
        else:
            new_names.append(name)
    return new_names

### Prepping dataframe (without species)
sim_bact_sorted_df.to_csv(data_prod_path+"sim_bact_sorted_df.csv")
sim_bact_sorted_data_df = sim_bact_sorted_df.drop(index="Species").drop(columns=["Species"])
#display(sim_bact_sorted_data_df)
sim_bact_sorted_data_df = sim_bact_sorted_data_df.astype(float) 

### Making unique labels
print(sim_bact_sorted_df["Species"].values)
labels_abbrev = reformat_bact_names(sim_bact_sorted_df["Species"].values)
labels_unique = unique_axis_labels(labels_abbrev)

plt.figure(figsize=(12,8))
sns.heatmap(sim_bact_sorted_data_df, cmap="YlGnBu")

plt.xticks(
    ticks=range(len(sim_bact_sorted_data_df.index)+1),
    labels=labels_unique,
    rotation=90,
    fontsize=6
    )

plt.yticks(
    ticks=range(len(sim_bact_sorted_data_df.index)+1),
    labels=labels_unique,
    fontsize=6
    )

# Adding y lines for species 
y_labels = sim_bact_sorted_df["Species"].values # Get the species labels for the y-axis
species_change_indices_y = [i for i in range(1, len(y_labels)) if y_labels[i] != y_labels[i-1]] # Find the indices where the species label changes
for idy in species_change_indices_y: # Plot horizontal lines at these indices
    plt.hlines(idy-1, xmin=0, xmax=len(sim_bact_sorted_data_df.columns), colors='grey', linestyles='dashed', linewidth=0.5)

# Adding x lines for species 
x_labels = sim_bact_sorted_df["Species"].values # Get the species labels for the y-axis
species_change_indices_x = [i for i in range(1, len(x_labels)) if x_labels[i] != x_labels[i-1]] # Find the indices where the species label changes
for idx in species_change_indices_x: # Plot horizontal lines at these indices
    plt.vlines(idx-1, ymin=0, ymax=len(sim_bact_sorted_data_df.columns), colors='grey', linestyles='dashed', linewidth=0.5)

plt.title(f"Similarity of bacterial strains by family with {K}mer")

#### Creating Sequence Bloom Tree (SBT) using Sourmash

In [ ]:
import sourmash, os, sys
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"
minhashes = []
sig_dir = data_prod_path+"bact_minhash_37/"
print("Number of signatures", len(os.listdir(sig_dir)))

### Initializing tree
import sourmash.sbtmh
tree = sourmash.sbtmh.create_sbt_index()

In [ ]:
from sourmash.sbtmh import SigLeaf
for filename in os.listdir(sig_dir):
    sig = sourmash.load_one_signature(sig_dir+filename, ksize=37)
    leaf = SigLeaf(sig.md5sum(), sig)
    tree.add_node(leaf)

filename = tree.save(data_prod_path + '/bact37.sbt.zip')

#### Searching the SBT

In [ ]:
import sourmash, os, sys
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

tree = sourmash.load_file_as_index(data_prod_path + '/bact37.sbt.zip')

In [ ]:
### Query a DNA sequence
import screed
filename = raw_data_path + "phagehost_KU/bacteria_fasta/J1_21_reoriented.fna"
query_seq = next(iter(screed.open(filename))).sequence
print(f'got {len(query_seq)} DNA characters to query')

### Creating MinHash for query sequence
K=37
minhash = sourmash.MinHash(ksize=37, n=1000, scaled=0)
minhash.add_sequence(query_seq)

query_sig = sourmash.SourmashSignature(minhash, name='my favorite query')

In [ ]:
### Searching the SBT
for similarity, found_sig, filename in tree.search(query_sig, threshold=0.1):
   print(query_sig)
   print(found_sig)
   print(similarity)

## Coloring mrbayes consensus tree based on phage familiy

### Reading the tree using Bio.Phylo

In [ ]:
import os
from Bio import Phylo
import matplotlib.pyplot as plt
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

# filename (adjust if located in a different folder)
filename = "phageKU_aligned.nexus.con.tre"
mrbayes_dir = "mrbayes_phageKU/"

dir_path = os.path.join(data_prod_path, mrbayes_dir)
if not os.path.exists(dir_path):
    raise FileNotFoundError(dir_path)

file_path = os.path.join(dir_path, filename)
if not os.path.exists(file_path):
    raise FileNotFoundError(file_path)

# try common tree formats until one succeeds; use parse() to handle files with multiple trees and take the first one
for fmt in ("nexus", "newick", "nexml", "phyloxml"):
    try:
        trees = Phylo.parse(file_path, fmt)
        tree = next(trees)  # get the first tree from the file
        print(f"Loaded first tree from {file_path} (format='{fmt}'). Terminals: {len(tree.get_terminals())}")
        break
    except StopIteration:
        print(f"No trees found in {file_path} with format='{fmt}'")
    except Exception as e:
        print(f"Could not parse {file_path} with format='{fmt}': {e}")

if 'tree' not in locals():
    raise ValueError(f"Could not read any tree from {file_path} with known formats.")

### Coloring dictionary

In [ ]:
import seaborn as sns

### Extract labels from the tree
labels = [term.name for term in tree.get_terminals()]

# create a color dictionary based on the first part (prefix before first "_") of each label
prefixes = [lab.split("_")[0] for lab in labels]
unique_prefixes = list(dict.fromkeys(prefixes))  # preserve order of first appearance

# choose a more distinct, colorblind-friendly palette (fall back to HUSL for many categories)
n = max(2, len(unique_prefixes))
if n <= 8:
    palette = sns.color_palette("colorblind", n_colors=n)
else:
    palette = sns.husl_palette(n_colors=n, h=1.0, s=0.9, l=0.6)

def _rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(*(int(255 * c) for c in rgb))

color_dict = {p: _rgb_to_hex(palette[i % len(palette)]) for i, p in enumerate(unique_prefixes)}

# optional: map each full label to its color (useful for plotting)
label_color_map = {lab: color_dict[lab.split("_")[0]] for lab in labels}

print("Prefix -> color:", color_dict)
print("Label -> color (sample):", dict(list(label_color_map.items())[:5]))

### Visualize the tree

In [ ]:
# visualize the loaded tree (uses existing variables: tree, plt, Phylo, data_prod_path, mrbayes_dir, filename)
# ladderize for nicer layout, plot, and save to file
tree.ladderize()

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(1, 1, 1)

# draw tree onto the axis without immediately showing (so we can adjust)
Phylo.draw(tree, axes=ax, show_confidence=False, do_show=False, label_colors=label_color_map)

ax.set_title(f"MrBayes Phylogenic Tree of Phages", fontsize=14)
#plt.tight_layout()
ax.set_xlim(-0.025, 2.25)

# save figure next to the tree file
out_dir = os.path.join(data_prod_path, mrbayes_dir)
out_path = os.path.join(out_dir, "phage_tree.png")
fig.savefig(out_path, dpi=300)
print(f"Saved tree plot to: {out_path}")

plt.show()

## Constructing mulitple Phage & Bact signature dirs

In [ ]:
from manipulations import construct_SM_sketches
import os, sys 
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

for k in [6, 9, 12, 15, 18, 24]:
    for n in [50, 100, 500, 1000, 5000]:
        #Phages minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                            k = k, 
                            outdir = f"PhageMinhash_n{n}_k{k}_rev/", 
                            quiet = False,
                            sourmash_parameters=[n, 0],
                            include_reverse=True)
        
        #Bacteria minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", 
                        k = k, 
                        outdir = f"BactMinhash_n{n}_k{k}_rev/", 
                        quiet = False,
                        sourmash_parameters=[n, 0],
                        include_reverse=True)

## Defining clusters based on similarity - Bact
Defining bacteria and phage cluster based on sequence (jaccard) similarity - only downsampled genomes

### Load minhash signatures and calculate jaccard similarity

In [ ]:
import pandas as pd
import sourmash, os, sys
from paths import *
K = 12
n = 500
minhashes = []
print(f"Raw data path: {raw_data_path}\nData production path: {data_prod_path}")
sig_dir = data_prod_path+"SM_sketches/"+f"BactMinhash_n{n}_k{K}/"
print("Number of signatures", len(os.listdir(sig_dir)))

"""
### Loading sigs - individually
for file in os.listdir(sig_dir):
    try:
        single_sig = sourmash.load_one_signature(sig_dir+file)
        print(single_sig)
    except: #catching wrongful load of signatures
        print(f"Failed to load {file}, now exiting")
        sys.exit(1)
    minhashes.append(single_sig)
"""

### Loading sigs - collectively
loaded_sigs = list(sourmash.load_file_as_signatures(sig_dir))
print(loaded_sigs[0], loaded_sigs[1])

In [ ]:
from tqdm import tqdm
import numpy as np

num_sigs = len(loaded_sigs)
count_zero_jac = 0
count_nonzero_jac = 0
jac_matrix = np.zeros((num_sigs, num_sigs))

for i in tqdm(range(num_sigs), desc="Iterating through loaded sigs (Outer loop)", unit="sigs"):
    for j in range(num_sigs):
        jac = loaded_sigs[i].jaccard(loaded_sigs[j])
        if jac > 0:
            jac_matrix[i, j] = jac
            count_nonzero_jac += 1
        else:
            #no need to add at jac_matrix, as it is already filled with zeros
            count_zero_jac += 1

print(f"Jaccard sim above 0 vs at zero: {count_nonzero_jac}/{count_zero_jac}")

In [ ]:
#print([l.minhash.hashes.keys() for l in loaded_sigs])

bact_names = []
for i in range(num_sigs):
    #print(str(loaded_sigs[i]), type(loaded_sigs[i]))
    bact_names.append(str(loaded_sigs[i]))

for name in bact_names:
    print(name, type(name))

#### Similarity plot ordered by family
Lookup ids in excel file

In [ ]:
hostrange_pdf = pd.read_excel(raw_data_path+"phagehost_KU/Hostrange_data_all_crisp_iso.xlsx", sheet_name="sum_hostrange", header=1)
id_lookup_bact = hostrange_pdf.set_index("Seq ID")["Species"] #fasta seq IDs + bacteria species lookup
id_lookup_bact.index = id_lookup_bact.index.str.replace("_reoriented", "", regex=False) # Adjusting index to match the format of bact_names (removing "_reoriented" suffix)
print(id_lookup_bact)

sim_bact_df = pd.DataFrame(jac_matrix)
sim_bact_df.columns = bact_names
sim_bact_df.index = bact_names
#display(sim_bact_df)

sim_bact_df = sim_bact_df.join(id_lookup_bact, how="left")
sim_bact_df["Species"] #Species is now in sim_bact_df columns
display(sim_bact_df["Species"])
sim_bact_df.loc['Species'] = list(sim_bact_df["Species"].values)+["NA"] # Add 'Species' as a row at the bottom of sim_bact_df

Sort index based on Species

In [ ]:
sim_bact_sorted_df = sim_bact_df.sort_values(by="Species", axis=0)
sim_bact_sorted_df["Species"]

In [ ]:
sim_bact_sorted_df

Transpose to sort again

In [ ]:
sim_bact_sorted_df = sim_bact_sorted_df.T
sim_bact_sorted_df = sim_bact_sorted_df.sort_values(by="Species", axis=0)
sim_bact_sorted_df #Columns and index positions match!

#### Plotting similarity with sorted indexes

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt 
import seaborn as sns

def unique_axis_labels(labels):
    counts = Counter(labels)
    l_dict = {}
    for l_uniq in set(labels):
        for i, label in enumerate(labels):
            if l_uniq == label:

                l_dict[l_uniq] = i + round(counts[l_uniq]/2)
                break
    labels_out = ["" for _ in range(len(labels))] #filled with "-" to remove duplicates
    for key, value in l_dict.items():
        labels_out[value] = key
    return labels_out


def reformat_bact_names(bact_names : list):
    new_names = []
    for name in bact_names:
        name_split = name.split(" ")
        if len(name_split) > 1: #two names
            kingdom_abbrev = name.split(" ")[0][0].strip(" ")
            new_names.append(kingdom_abbrev+". "+name.split(" ")[1])
        else:
            new_names.append(name)
    return new_names

### Prepping dataframe (without species)
sim_bact_sorted_df.to_csv(data_prod_path+"sim_bact_sorted_df.csv")
sim_bact_sorted_data_df = sim_bact_sorted_df.drop(index="Species").drop(columns=["Species"])
#display(sim_bact_sorted_data_df)
sim_bact_sorted_data_df = sim_bact_sorted_data_df.astype(float) 

### Making unique labels
print(sim_bact_sorted_df["Species"].values)
labels_abbrev = reformat_bact_names(sim_bact_sorted_df["Species"].values)
labels_unique = unique_axis_labels(labels_abbrev)

plt.figure(figsize=(12,8))
sns.heatmap(sim_bact_sorted_data_df, cmap="YlGnBu")

plt.xticks(
    ticks=range(len(sim_bact_sorted_data_df.index)+1),
    labels=labels_unique,
    rotation=90,
    fontsize=6
    )

plt.yticks(
    ticks=range(len(sim_bact_sorted_data_df.index)+1),
    labels=labels_unique,
    fontsize=6
    )

# Adding y lines for species 
y_labels = sim_bact_sorted_df["Species"].values # Get the species labels for the y-axis
species_change_indices_y = [i for i in range(1, len(y_labels)) if y_labels[i] != y_labels[i-1]] # Find the indices where the species label changes
for idy in species_change_indices_y: # Plot horizontal lines at these indices
    plt.hlines(idy-1, xmin=0, xmax=len(sim_bact_sorted_data_df.columns), colors='grey', linestyles='dashed', linewidth=0.5)

# Adding x lines for species 
x_labels = sim_bact_sorted_df["Species"].values # Get the species labels for the y-axis
species_change_indices_x = [i for i in range(1, len(x_labels)) if x_labels[i] != x_labels[i-1]] # Find the indices where the species label changes
for idx in species_change_indices_x: # Plot horizontal lines at these indices
    plt.vlines(idx-1, ymin=0, ymax=len(sim_bact_sorted_data_df.columns), colors='grey', linestyles='dashed', linewidth=0.5)

plt.title(f"Similarity of bacterial strains by family with {K}mer")
plt.savefig(data_prod_path+f"sim_bact_heatmap_n{n}_k{K}_rev.png", dpi=300)
plt.show()

### Defining clusters of bacteria

In [ ]:
cluster_mapping_bact = {
    0 : ["Pectobacterium atrosepticum"],
    1 : ["Pectobacterium brasiliense"],
    2 : ["Pectobacterium parmentieri"],
    3 : ["Pectobacterium polaris", "Pectobacterium carotovorum"],
    4 : ["Remaining species"]
}

In [ ]:
set(sim_bact_sorted_df["Species"])

##### Including binary host range for number of positive interactions per cluster

In [ ]:
from manipulations import hostrange_df_to_dict, binarize_host_range
from io_operations import call_hostrange_df

bact_lookup, host_range_df = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/Hostrange_data_all_crisp_iso.xlsx"))
host_range_data = binarize_host_range(hostrange_df_to_dict(host_range_df), continous=False)
host_range_data = {bact.replace("_reoriented", ""): interactions for bact, interactions in host_range_data.items()} # if "_reoriented" is in the bacteria names in host_range_data, remove it to match the bacteria names in the presence matrix.

#display(host_range_data)

# Obtain number of phages 
phage_count = len(host_range_data[next(iter(host_range_data))]) # Assuming all bacteria have the same number of phages, we can take the length of the interactions for any bacteria to get the number of phages.
print(f"Number of phages: {phage_count}")

# Obtain number of interactions for each bacteria
bact_int_sum = {}
for bact, phage in host_range_data.items():
    bact_pha_sum = 0 
    for pha in phage:
        interaction = host_range_data[bact][pha]
        #print(f"{bact} - {pha}: {interaction} interactions")
        bact_pha_sum += float(interaction)
    bact_int_sum[bact] = bact_pha_sum

In [ ]:
# Add a Cluster column based on Species values
species_to_cluster = {
    species: cluster
    for cluster, species_list in cluster_mapping_bact.items()
    for species in species_list
    if species != "Remaining species"
}

sim_bact_sorted_df["Cluster"] = (
    sim_bact_sorted_df["Species"]
    .map(species_to_cluster)
    .fillna(4)
    .astype(int)
)

# map per-strain interaction sums from bact_int_sum, then aggregate per cluster
sim_bact_sorted_df["interaction_sum"] = (
    sim_bact_sorted_df.index.to_series().map(bact_int_sum).fillna(0.0)
)

# remove metadata row if present before plotting
plot_df = sim_bact_sorted_df.loc[sim_bact_sorted_df.index != "Species"].copy()

plt.figure(figsize=(10, 6))

cluster_counts = plot_df["Cluster"].value_counts().sort_index()
cluster_interaction_sums = plot_df.groupby("Cluster")["interaction_sum"].sum().sort_index()
ax = sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette="Set2")

cluster_names = {
    cluster: ("Remaining species" if cluster == 4 else ", ".join(species_list))
    for cluster, species_list in cluster_mapping_bact.items()
}

for bar, cluster in zip(ax.patches, cluster_counts.index):
    num_entries = cluster_counts[cluster]*phage_count # multiplying by number of phages to get total interactions for the cluster
    height = bar.get_height()
    interaction_total = cluster_interaction_sums.get(cluster, 0.0)

    if "," in cluster_names.get(cluster, str(cluster)): # if there are multiple species in the cluster, split the label into two lines for better readability
        species_list = cluster_names[cluster].split(", ")
        label = "\n".join(species_list)
    else:
        label = cluster_names.get(cluster, str(cluster))

    # existing in-bar cluster label
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height / 2,
        label,
        ha="center",
        va="center",
        rotation=90,
        fontsize=8,
        color="black",
    )

    # new top annotation with summed interactions for the cluster
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + max(cluster_counts.values) * 0.01,
        f"PIC: {interaction_total:.0f} | PIF: {(interaction_total/num_entries)*100:.0f}%",
        ha="center",
        va="bottom",
        fontsize=8,
        color="black",
    )

ax.set_ylim(0, max(cluster_counts.values) * 1.2)
plt.xlabel("Cluster")
plt.ylabel("Number of entries")
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.ylim(0, max(cluster_counts.values) * 1.05)
# Make a custom legend that has PIC: "Positive Interaction Count" and PIF: "Positive Interaction Frequency"
plt.legend(
    handles=[
        plt.Line2D([0], [0], color="none", label="PIC: Positive Interaction Count"),
        plt.Line2D([0], [0], color="none", label="PIF: Positive Interaction Frequency (%)"),
    ],
    loc="upper right",
    fontsize=8,
)
plt.title("Cluster Assignment on Bacterial Sequence Similarity")
plt.show()
plt.savefig(os.path.join(data_prod_path, "cluster_barplot.png"), dpi=300)

#### Saving bacteria clusters

In [ ]:
bact_clusters_with_genus = sim_bact_sorted_df[["Cluster", "Species"]].copy()

# remove metadata row if present
bact_clusters_with_genus = bact_clusters_with_genus.loc[
    bact_clusters_with_genus.index != "Species"
]

bact_clusters_with_genus["genus"] = bact_clusters_with_genus["Species"].str.split().str[0]
bact_clusters_with_genus["strain_short"] = bact_clusters_with_genus.index
bact_clusters_with_genus.index = bact_clusters_with_genus.index + "_reoriented"
bact_clusters_with_genus = bact_clusters_with_genus[["Cluster", "strain_short", "Species", "genus"]]

display(bact_clusters_with_genus)
bact_clusters_with_genus.to_csv(data_prod_path+"bact_clusters_with_genus.csv", index=True)

## Defining clusters based on similarity - Phage
Defining bacteria and phage cluster based on sequence (jaccard) similarity - only downsampled genomes

### Load minhash signatures and calculate jaccard similarity

In [ ]:
import pandas as pd
import sourmash, os, sys
from paths import *
K = 12
n = 500
minhashes = []
print(f"Raw data path: {raw_data_path}\nData production path: {data_prod_path}")
sig_dir = data_prod_path+"SM_sketches/"+f"PhageMinhash_n{n}_k{K}/"
print("Number of signatures", len(os.listdir(sig_dir)))

"""
### Loading sigs - individually
for file in os.listdir(sig_dir):
    try:
        single_sig = sourmash.load_one_signature(sig_dir+file)
        print(single_sig)
    except: #catching wrongful load of signatures
        print(f"Failed to load {file}, now exiting")
        sys.exit(1)
    minhashes.append(single_sig)
"""

### Loading sigs - collectively
loaded_sigs = list(sourmash.load_file_as_signatures(sig_dir))
print(loaded_sigs[0], loaded_sigs[1])

In [ ]:
from tqdm import tqdm
import numpy as np

num_sigs = len(loaded_sigs)
count_zero_jac = 0
count_nonzero_jac = 0
jac_matrix = np.zeros((num_sigs, num_sigs))

for i in tqdm(range(num_sigs), desc="Iterating through loaded sigs (Outer loop)", unit="sigs"):
    for j in range(num_sigs):
        jac = loaded_sigs[i].jaccard(loaded_sigs[j])
        if jac > 0:
            jac_matrix[i, j] = jac
            count_nonzero_jac += 1
        else:
            #no need to add at jac_matrix, as it is already filled with zeros
            count_zero_jac += 1

print(f"Jaccard sim above 0 vs at zero: {count_nonzero_jac}/{count_zero_jac}")

In [ ]:
#print([l.minhash.hashes.keys() for l in loaded_sigs])

phage_names = []
for i in range(num_sigs):
    #print(str(loaded_sigs[i]), type(loaded_sigs[i]))
    phage_names.append(str(loaded_sigs[i]))

for name in phage_names:
    print(name, type(name))

#### Similarity plot ordered by family
Lookup ids in excel file

In [ ]:
# hostrange_pdf = pd.read_excel(raw_data_path+"phagehost_KU/Hostrange_data_all_crisp_iso.xlsx", sheet_name="sum_hostrange", header=1)
# id_lookup_bact = hostrange_pdf.set_index("Seq ID")["Species"] #fasta seq IDs + bacteria species lookup
# print(id_lookup_bact)

sim_phage_df = pd.DataFrame(jac_matrix)
sim_phage_df.columns = phage_names
sim_phage_df.index = phage_names
display(sim_phage_df)

### Defining clusters of Phages

In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Convert Similarity to Distance
# Assuming 'df' is your dataframe from the screenshot
# distance = 1 - similarity
dist_matrix = 1 - sim_phage_df

# 2. Apply Agglomerative Clustering
# We use 'precomputed' affinity because we are providing a distance matrix
cluster_model = AgglomerativeClustering(
    n_clusters=5, 
    metric='precomputed', 
    linkage='average' # 'average' or 'complete' work well for genomic clusters
)

clusters = cluster_model.fit_predict(dist_matrix)

# 3. Map clusters back to your sequence names
results = pd.DataFrame({
    'Sequence': sim_phage_df.index,
    'Cluster': clusters
}).sort_values(by='Cluster')

print(results)

In [ ]:
import pandas as pd
import numpy as np

# 1. Calculate the distance matrix
dist_matrix = 1 - sim_phage_df

# 2. Get a 1D representation for sorting
# We calculate the mean distance of each sequence to all others.
# Lower mean distance = more "central" sequences.
# Higher mean distance = more "outlier" sequences.
sorting_metric = dist_matrix.mean(axis=1)

# 3. Sort the sequences based on this metric
sorted_sequences = sorting_metric.sort_values().index

# 4. Divide into 5 equal groups
# Using numpy.array_split to handle cases where the total count 
# isn't perfectly divisible by 5.
groups = np.array_split(sorted_sequences, 5)

# 5. Map back to a clean DataFrame
cluster_map = {}
for i, group in enumerate(groups):
    for seq in group:
        cluster_map[seq] = {i}

results = pd.DataFrame.from_dict(cluster_map, orient='index', columns=['Cluster'])

print(f"Total sequences: {len(results)}")
print(results['Cluster'].value_counts()) # Verify equal sizes
print(results)

In [ ]:
from manipulations import hostrange_df_to_dict, binarize_host_range
from io_operations import call_hostrange_df

bact_lookup, host_range_df = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/Hostrange_data_all_crisp_iso.xlsx"))
host_range_data = binarize_host_range(hostrange_df_to_dict(host_range_df), continous=False)
host_range_data = {bact.replace("_reoriented", ""): interactions for bact, interactions in host_range_data.items()} # if "_reoriented" is in the bacteria names in host_range_data, remove it to match the bacteria names in the presence matrix.

# Obtain number of phages 
bact_count = len(host_range_data) # Assuming all bacteria have the same number of phages, we can take the length of the interactions for any bacteria to get the number of phages.
print(f"Number of bacteria: {bact_count}")

# Flip host_range_data to have phages as keys and bacteria as subkeys, to obtain number of interactions for each phage
phage_host_range_data = {}
for bact, phage in host_range_data.items():
    for pha, interaction in phage.items():
        if pha not in phage_host_range_data:
            phage_host_range_data[pha] = {}
        phage_host_range_data[pha][bact] = interaction

# Obtain number of interactions for each phage
phage_int_sum = {}
for pha, bact_dict in phage_host_range_data.items():
    phage_pha_sum = 0
    for bact, interaction in bact_dict.items():
        phage_pha_sum += float(interaction)
    phage_int_sum[pha] = phage_pha_sum

#### Saving bacteria clusters

In [ ]:
phage_clusters = results.copy()

# remove metadata row if present
phage_clusters = phage_clusters.loc[
    phage_clusters.index != "Species"
]

phage_clusters.index = phage_clusters.index.str.split("_").str[-1] # Convert index to strain short names
value_counts = phage_clusters['Cluster'].value_counts().sort_index() # Making a count of cluster assignment

# Build per-phage positive interaction sums from phage_host_range_data
phage_sum_lookup = {}
for pha, bact_dict in phage_host_range_data.items():
    pha_short = str(pha).split("_")[-1]
    phage_sum_lookup[pha_short] = phage_sum_lookup.get(pha_short, 0.0) + sum(float(v) for v in bact_dict.values())

# Map interaction sums to clustered phages
phage_clusters["interaction_sum"] = phage_clusters.index.to_series().map(phage_sum_lookup).fillna(0.0)

# Aggregations per cluster
cluster_counts = phage_clusters["Cluster"].value_counts().sort_index()
cluster_interaction_sums = phage_clusters.groupby("Cluster")["interaction_sum"].sum().sort_index()
cluster_phages = phage_clusters.groupby("Cluster").apply(lambda x: ", ".join(x.index)).to_dict()

# plot as barplot
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette="Set2")

# use "-".join(phages.split(", ")) as label for each bar
for bar, cluster in zip(plt.gca().patches, value_counts.index):
    height = bar.get_height()
    phages = cluster_phages[cluster]
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height / 2,
        "-".join(phages.split(", ")),
        ha="center",
        va="center",
        rotation=90,
        fontsize=8,
        color="black",
    )

    interaction_total = cluster_interaction_sums.get(cluster, 0.0)
    denom = cluster_counts.loc[cluster] * bact_count
    pif = (interaction_total / denom * 100.0) if denom else 0.0
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(cluster_counts.values) * 0.01,
        f"PIC: {interaction_total:.0f} | PIF: {pif:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.xlabel("Cluster")
plt.ylabel("Number of sequences")
plt.title("Cluster Assignment on Phage Sequence Similarity")
# Make a custom legend that has PIC: "Positive Interaction Count" and PIF: "Positive Interaction Frequency"
plt.legend(
    handles=[
        plt.Line2D([0], [0], color="none", label="PIC: Positive Interaction Count"),
        plt.Line2D([0], [0], color="none", label="PIF: Positive Interaction Frequency (%)"),
    ],
    loc="upper right",
    fontsize=8,
)

plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.show()
plt.savefig(os.path.join(data_prod_path, f"phage_clusters_k{K}_n{n}.png"), dpi=300)

display(phage_clusters)
#phage_clusters.to_csv(data_prod_path+"phage_clusters.csv", index=True)

## Hierarchical clustering using sketches

Through a combination of sourmash compare and sourmash plot, similarity matrix plots with dendograms could be constructed for each downsampling method. Using the plug-in sourmash_plugin_betterplot for better plots.

### Example compare:
> sourmash compare BactMinhash_n500_k12/ -o /home/projects/s215045/PredictPhagePPI/data_prod/SM_sketches/sim_matrices/BactSim_n500_k12.mat --labels-to /home/projects/s215045/PredictPhagePPI/data_prod/SM_sketches/sim_matrices/BactSim_n500_k12.mat.labels_to.csv

### Example plot (without plug-in)
> sourmash plot PhageSim_n500_k12.mat

### Example dendogram plot with plug-in
converting bact labels to labels with genus names
> /home/projects/s215045/PredictPhagePPI/scripts/annotation_scripts/prefix_bact_labels.sh BactSim_n500_k12.mat.labels_to.csv /home/projects/s215045/PredictPhagePPI/data_prod/bact_clusters_with_genus.csv > BactSim_n500_k12.mat.labels_prefixed.csv

plotting bacts
> sourmash scripts plot2 BactSim_n500_k12.mat BactSim_n500_k12.mat.labels_prefixed.csv -o /home/projects/s215045/PredictPhagePPI/data_prod/SM_sketches/sim_matrices/BactDendro_n500_k12.png --cut-point=1.08 --cluster-out --figsize-y 18 --figsize-x 20

plotting phages
> sourmash scripts plot2 PhageSim_n500_k12.mat PhageSim_n500_k12.mat.labels_to.csv -o /home/projects/s215045/PredictPhagePPI/data_prod/SM_sketches/sim_matrices/PhageDendro_n500_k12.png --cut-point=1.12 --cluster-out --figsize-y 18 --figsize-x 20


The results were saved in sim_matrices/ in the downsampling folder (SM_sketches / encoded_sketches) located in data_prod/

#### See what compare create (type of distance in matrix)

In [1]:
from pathlib import Path
import numpy as np
import pickle

import scipy.io

mat_path = Path("/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_test/sim_matrices/BactSim_n500_k12.mat")
if not mat_path.exists():
    raise FileNotFoundError(mat_path)

data = None
loader = None

# try numpy.load
try:
    data = np.load(mat_path, allow_pickle=True)
    loader = "numpy.load"
except Exception as e_np:
    try:
        data = scipy.io.loadmat(mat_path)
        loader = "scipy.io.loadmat"
    except Exception as e_scipy:
        try:
            with open(mat_path, "rb") as fh:
                data = pickle.load(fh)
            loader = "pickle.load"
        except Exception as e_pickle:
            raise RuntimeError(f"Failed to load file: numpy error: {e_np}; scipy error: {e_scipy}; pickle error: {e_pickle}")

print("Loaded with:", loader, "type:", type(data))

# normalize to a single ndarray if possible
arr = None
if isinstance(data, np.lib.npyio.NpzFile):
    keys = list(data.keys())
    print("npz keys:", keys)
    arr = data[keys[0]]
elif isinstance(data, dict):
    # scipy.loadmat returns dict; choose the largest ndarray value
    nd_arrays = {k: v for k, v in data.items() if isinstance(v, np.ndarray)}
    if nd_arrays:
        arr = max(nd_arrays.values(), key=lambda x: x.size)
    else:
        arr = data
else:
    arr = data

if isinstance(arr, np.ndarray):
    print("ndarray shape:", arr.shape, "dtype:", arr.dtype)
    try:
        print("min, max, mean:", np.nanmin(arr), np.nanmax(arr), np.nanmean(arr))
    except Exception:
        pass
    if arr.ndim == 2 and arr.shape[0] == arr.shape[1]:
        print("Symmetric (allclose to transpose):", np.allclose(arr, arr.T))
else:
    print("Loaded object is not an ndarray. repr:", repr(arr)[:500])

FileNotFoundError: /home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_test/sim_matrices/BactSim_n500_k12.mat

### Use sourmash to read signature (check for error)

In [4]:
from paths import *
import sourmash
n = 500
k = 12

# Example: Load a signature and print its details
sig_dir = data_prod_path+"encoded_sketches_test/"+f"Bactmmh3_n{n}_k{k}/"
example_sig = sourmash.load_file_as_signatures(sig_dir + os.listdir(sig_dir)[0])
print(example_sig)

<generator object MultiIndex.signatures at 0x736be2b9cee0>


### Collecting cluster files to one combined
save to data_prod

In [6]:
import os
import pandas as pd
import sys
from paths import *
import re
from io_operations import call_hostrange_df

def collect_sourmash_clusters(sim_mat_dir:str, n:int, k:int, save_dir:str = None):
    """
    Collects sourmash cluster files from the specified directory, assigns cluster labels based on file naming conventions, 
    and combines them into two dataframes: one for bacteria and one for phages. Optionally saves the combined dataframes to CSV files in the specified save directory.
    
    Args:
        sim_mat_dir (str): Directory containing the cluster matrix CSV files.
        n (int): The 'n' parameter used in the sourmash sketches, for reference in file naming.
        k (int): The 'k' parameter used in the sourmash sketches, for reference in file naming.
        save_dir (str, optional): Directory to save the combined dataframes as CSV files. If None, dataframes are not saved. Defaults to None.
    
    Returns:
        combined_bact (pd.DataFrame): Combined dataframe containing bacterial cluster information.
        combined_phage (pd.DataFrame): Combined dataframe containing phage cluster information.
    """

    combined_bact = pd.DataFrame()
    combined_phage = pd.DataFrame()
    bact_clust_count = 0
    phage_clust_count = 0
    n_string = f"n{n}"
    k_string = f"k{k}"
    pattern = re.compile(r'.*\d+\.csv$')

    data2 = False
    OHE = False
    if "_data2" in sim_mat_dir:
        data2 = True
    if "encoded" in sim_mat_dir:
        OHE = True

    if data2:
        bact_lookup, _ = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/data2_EOP.xlsx"), sheet_name="Sheet1", data2=True)
        bact_lookup = {k: (v.split()[0][0] + v.split()[1] if len(v.split()) > 1 else v) for k, v in bact_lookup.items()}

    else:
        bact_lookup, _ = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/Hostrange_data_all_crisp_iso.xlsx"))
        bact_lookup = {k: (v.split()[0][0] + "._" + v.split()[1] if len(v.split()) > 1 else v) for k, v in bact_lookup.items()}
    bact_set = set(bact_lookup.values())

    for file in os.listdir(sim_mat_dir):
        if not pattern.match(file):
            continue
        if n_string not in file or k_string not in file:
            continue

        try:
            sim_mat = pd.read_csv(os.path.join(sim_mat_dir, file), index_col=0)
            #print(f"Loaded similarity matrix from {file} with shape {sim_mat.shape}")
        except Exception as e:
            print(f"Failed to load {file}: {e}")
            sys.exit(1)

        if "Phage" in file:
            sim_mat = sim_mat[["label"]]
            sim_mat["host"] = sim_mat["label"].str.split("_").str[0] # Extract host name 
            sim_mat["label"] = sim_mat["label"].str.split("_").str[-1] # Convert label to phage only name
            sim_mat["Cluster"] = phage_clust_count
            sim_mat.set_index("label", inplace=True)
            combined_phage = pd.concat([combined_phage, sim_mat], axis=0)
            phage_clust_count += 1

        elif "Bact" in file:
            sim_mat = sim_mat[["label"]]
            #if any element in sim_mat["label"] starts with any element in bact_set, remove the prefix bact_set element
            sim_mat["label"] = sim_mat["label"].apply(
                lambda x: next(
                    (x[len(prefix):] for prefix in sorted(bact_set, key=len, reverse=True) if x.startswith(prefix)),
                    x
                )
            )
            sim_mat["label"] = sim_mat["label"].str.replace("_bp=", "", regex=False)#remove "=" from sim_mat["label"]
            sim_mat["label"] = sim_mat["label"].str.lstrip("_")#if sim_mat["label"] starts with "_" remove it
            sim_mat["Cluster"] = bact_clust_count
            sim_mat.set_index("label", inplace=True)
            # Create new column "Species" by mapping the index (strain names) to species names using bact_lookup
            if data2:
                sim_mat["Species"] = "Host " + sim_mat.index.str.split("_").str[1].str.lstrip("KU")
            else:
                sim_mat["Species"] = sim_mat.index.to_series().map(
                    lambda x: bact_lookup.get(x, bact_lookup.get(f"{x}_reoriented", "Unknown"))
                )
            combined_bact = pd.concat([combined_bact, sim_mat], axis=0)
            bact_clust_count += 1

    if save_dir is not None:
        combined_bact.to_csv(os.path.join(save_dir, "combined_bact_clusters.csv"))
        combined_phage.to_csv(os.path.join(save_dir, "combined_phage_clusters.csv"))

    return combined_bact, combined_phage


SM_sim_mat_dir = data_prod_path+"SM_sketches/"+"sim_matrices/"
encoded_sim_mat_dir = data_prod_path+"encoded_sketches_data2/"+"sim_matrices/"

#SM_bact, SM_phage = collect_sourmash_clusters(SM_sim_mat_dir)
encoded_bact, encoded_phage = collect_sourmash_clusters(encoded_sim_mat_dir, n=500, k=12, save_dir=data_prod_path+"encoded_sketches_data2/")

print(encoded_bact)
print(encoded_phage)
        

Empty DataFrame
Columns: []
Index: []
Empty DataFrame
Columns: []
Index: []


In [52]:
SM_sim_mat_dir = data_prod_path+"SM_sketches/"+"sim_matrices/"
encoded_sim_mat_dir = data_prod_path+"encoded_sketches_data2/"+"sim_matrices/"

#SM_bact, SM_phage = collect_sourmash_clusters(SM_sim_mat_dir)
encoded_bact, encoded_phage = collect_sourmash_clusters(encoded_sim_mat_dir, n=500, k=12, save_dir=data_prod_path+"encoded_sketches_data2/")

display(encoded_bact)
display(encoded_phage)
#SM_bact.to_csv(data_prod_path+"BactSim_n500_k12_combined.csv", index=True)

{'H1', 'H7', 'H6', 'H12', 'H4', 'H3', 'H9', 'H10', 'H11', 'H5', 'H13', 'H2', 'H8'}
Index(['KU7', 'KU13', 'KU2', 'KU1', 'KU11', 'KU5', 'KU12', 'KU3'], dtype='str', name='label')
Index(['KU4', 'KU9', 'KU10', 'KU6', 'KU8', 'KU6'], dtype='str', name='label')


,Cluster,Species
label,,
Kp_KU7_circular206344_linear5324540,0,Host 7
Kp_KU13_circular5717070_linear0,0,Host 13
Kp_KU2_circular5363689_linear0,0,Host 2
Kp_KU1_circular5445670_linear0,0,Host 1
Kp_KU11_circular5649217_linear0,0,Host 11
Kp_KU5_circular197875_linear5275929,0,Host 5
Kp_KU12_circular68599_linear5202832,0,Host 12
Kp_KU3_circular213113_linear5103018,0,Host 3
Kp_KU4_circular102829_linear2630891,1,Host 4


,host,Cluster
label,,
11,Nepotimus,0
6,Heraclius,1
9,Quartinus,2
1,Ravello,3
13,Silbannacus,4
4,Phocas,5
6,Anivius,6
13,Volusianus,7
13,Balder,7


In [15]:
bact_lookup, host_range_df = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/data2_EOP.xlsx"), sheet_name="Sheet1", data2=True)
print("Original bact_lookup:", bact_lookup)
bact_lookup = {k: (v.split()[0][0] + v.split()[1] if len(v.split()) > 1 else v) for k, v in bact_lookup.items()}
display(host_range_df)

Original bact_lookup: {'Host 1': 'Host 1', 'Host 2': 'Host 2', 'Host 3': 'Host 3', 'Host 4': 'Host 4', 'Host 5': 'Host 5', 'Host 6': 'Host 6', 'Host 7': 'Host 7', 'Host 8': 'Host 8', 'Host 9 ': 'Host 9 ', 'Host 10': 'Host 10', 'Host 11': 'Host 11', 'Host 12 ': 'Host 12 ', 'Host 13': 'Host 13'}


,phage,Grebano_1_host1,Ravello_2_host1,Etui_3_host1,Maxentius_4_host2,Licinius_5_host2,Jovian_6_host2,Arcadius_7_host2,Avitus_8_host3,Marcian_9_host3,...,Trebonianus_41_host11,Skandal_42_host13,Balder_43_host13,Herennius_44_host13,Silbannacus_45_host13,Volusianus_46_host13,Galleinus_47_host13,Salolinus_48_host13,Carinus_49_host13,Galerius_50_host13
0,Host 1,1.000000,1.000000,1.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
1,Host 2,0.000000,0.000000,0.0000,1.000000,1,1.00,1.000000,0.047619,0.000061,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
2,Host 3,0.000000,0.000000,0.0000,0.076087,0,0.44,0.102222,1.000000,1.000000,...,0.000000,0.967742,0.000636,0.635135,0.000671,0.000825,0.010000,0.000609,0.002632,0.0014
3,Host 4,0.733333,0.000000,0.7000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
4,Host 5,0.000217,0.000008,0.0275,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
5,Host 6,0.000000,0.000000,0.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.003226,0.909091,0.000743,0.002429,0.000000,0.373333,0.260870,0.438596,0.0000
6,Host 7,0.000000,0.000000,0.0000,0.119565,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
7,Host 8,0.000000,0.000000,0.0000,0.295714,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
8,Host 9,0.000000,0.000000,0.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000032,0.008409,0.004459,0.001286,0.000000,0.000000,0.000000,0.002632,0.0000
9,Host 10,0.000000,0.000000,0.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000


In [21]:
bact_lookup, host_range_df = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/Hostrange_data_all_crisp_iso.xlsx"), sheet_name="sum_hostrange")
print("Original bact_lookup:", bact_lookup)
#bact_lookup = {k: (v.split()[0][0] + v.split()[1] if len(v.split()) > 1 else v) for k, v in bact_lookup.items()}
display(host_range_df)

Original bact_lookup: {'J14_21_reoriented': 'Acinetobacter calcoaceticus', 'J53_21_reoriented': 'Acinetobacter calcoaceticus', 'J105_22_reoriented': 'Chishuiella', 'J46_21_reoriented': 'Chryseobacterium', 'J50_21_reoriented': 'Chryseobacterium', 'J2264_1_22_KMC_reoriented': 'Chryseobacterium', 'J2264_3_22_KMC_reoriented': 'Chryseobacterium', 'J63_22_reoriented': 'Chryseobacterium', 'J64_22_reoriented': 'Chryseobacterium', 'J1_21_reoriented': 'Lelliottia', 'J91_22_reoriented': 'Lelliottia', 'J51_21_reoriented': 'Morganella morganii', 'J57_21_reoriented': 'Morganella morganii', 'J10_21_reoriented': 'Pectobacterium atrosepticum', 'J11_21_reoriented': 'Pectobacterium atrosepticum', 'J126_23_reoriented': 'Pectobacterium atrosepticum', 'J12_21_reoriented': 'Pectobacterium atrosepticum', 'J16_21_reoriented': 'Pectobacterium atrosepticum', 'J22_21_reoriented': 'Pectobacterium atrosepticum', 'J28_21_reoriented': 'Pectobacterium atrosepticum', 'J33_21_reoriented': 'Pectobacterium atrosepticum', 

,phage,Ymer,Taid,Poppous,Koroua,Abuela,Amona,Sabo,Mimer,Crus,...,Vims,Echoes,Galvinrad,Uther,Rip,Rup,Slaad,Pantea,Rap,Zann
0,J14_21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,J53_21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J105_22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,J46_21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,J50_21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,J109_23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,J101_22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,J15_21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,800000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,J4_21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Isolation host,Isolation host,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
hostrange_pdf = pd.read_excel(raw_data_path+"phagehost_KU/Hostrange_data_all_crisp_iso.xlsx", sheet_name="sum_hostrange", header=1)
display(hostrange_pdf)

# id_lookup_bact = hostrange_pdf[["Seq ID", "Species"]].rename(columns={"Seq ID": "Bacterium_Name"})
# display(id_lookup_bact)

# bact_lookup_df = pd.DataFrame(bact_lookup.items(), columns=["Bacterium_Name", "Species"])
# display(bact_lookup_df)

# hostrange_pdf = pd.read_excel(raw_data_path+"phagehost_KU/data2_EOP.xlsx", sheet_name="Sheet1", header=1)
# display(hostrange_pdf)

,isolate ID,Seq ID,Species,Hostrange_analysis,Phage,Ymer,Taid,Poppous,Koroua,Abuela,...,Vims,Echoes,Galvinrad,Uther,Rip,Rup,Slaad,Pantea,Rap,Zann
0,5.2,J14_21_reoriented,Acinetobacter calcoaceticus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41.1,J53_21_reoriented,Acinetobacter calcoaceticus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,96.1,J105_22_reoriented,Chishuiella,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,35.1,J46_21_reoriented,Chryseobacterium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,38.1,J50_21_reoriented,Chryseobacterium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,100A,J109_23_reoriented,Pseudomonas chlororaphis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,92.2,J101_22_reoriented,Pseudomonas marginalis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,6.1,J15_21_reoriented,Serratia liquefaciens,2022_mmr,+,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,800000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,2.4,J4_21_reoriented,Serratia plymuthica,2022_mmr_isolation,+,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Isolation host,Isolation host,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
from manipulations import hostrange_df_to_dict, binarize_host_range
host_range_data = binarize_host_range(hostrange_df_to_dict(host_range_df), continous=False)
host_range_data = {bact.replace("_reoriented", ""): interactions for bact, interactions in host_range_data.items()} # if "_reoriented" is in the bacteria names in host_range_data, remove it to match the bacteria names in the presence matrix.
print(host_range_data)

{'Host 1': {'Grebano_1_host1': 1, 'Ravello_2_host1': 1, 'Etui_3_host1': 1, 'Maxentius_4_host2': 0, 'Licinius_5_host2': 0, 'Jovian_6_host2': 0, 'Arcadius_7_host2': 0, 'Avitus_8_host3': 0, 'Marcian_9_host3': 0, 'Libius_10_host3': 0, 'Anthemius_11_host3': 0, 'Olybrius_12_host3': 0, 'Phocas_13_host4': 0, 'Caracalla_14_host4': 0, 'Geta_14_host4': 0, 'Leonitus_15_host4': 0, 'Artabasdos_16_host5': 1, 'Rangabe_17_host5': 1, 'Staurakios_18_host5': 1, 'Bardicus_19_host5': 1, 'Quintillus_20_host6': 0, 'Heraclius_21_host6': 0, 'Heraclonas_22_host6': 0, 'Anivius_23_host6': 0, 'Komnenos_24_host7': 0, 'Eudokia_25_host7': 0, 'Doukas_26_host7': 0, 'Arruntis_27_host7': 0, 'Hostillian_28_host8': 0, 'Pacatian_nan_host8': 0, 'Quartinus_29_host9': 0, 'Bonosus_30_host9': 0, 'Rozzorie_31_host10': 0, 'Brede_32_host10': 0, 'Didius_33_host10': 0, 'Septimius_34_host10': 0, 'Diadumenian_35_host10': 0, 'Elagabalus_36_host10': 0, 'Pius_37_host10': 0, 'Pupienus_38_host11': 0, 'Nepotimus_38_host11': 0, 'Balbinus_39_ho

#### Bact_clusters lookup

In [26]:
bact_clusters = pd.read_csv(os.path.join(data_prod_path, "bact_clusters_with_genus.csv"), index_col=0)
display(bact_clusters)

,Cluster,strain_short,Species,genus
J53_21_reoriented,4,J53_21,Acinetobacter calcoaceticus,Acinetobacter
J14_21_reoriented,4,J14_21,Acinetobacter calcoaceticus,Acinetobacter
J105_22_reoriented,4,J105_22,Chishuiella,Chishuiella
J2264_1_22_KMC_reoriented,4,J2264_1_22_KMC,Chryseobacterium,Chryseobacterium
J50_21_reoriented,4,J50_21,Chryseobacterium,Chryseobacterium
...,...,...,...,...
J109_23_reoriented,4,J109_23,Pseudomonas chlororaphis,Pseudomonas
J101_22_reoriented,4,J101_22,Pseudomonas marginalis,Pseudomonas
J15_21_reoriented,4,J15_21,Serratia liquefaciens,Serratia
J4_21_reoriented,4,J4_21,Serratia plymuthica,Serratia


In [30]:
prefix = "encoded_sketches_data2"
n = 500
k = 12
bact_clusters = pd.read_csv(os.path.join(data_prod_path, f"{prefix}", "sim_matrices",f"combined_bact_clusters_n{n}_k{k}.csv"), index_col=0)
display(bact_clusters)

,Cluster,Partition
label,,
Kp_KU4_circular102829_linear2630891,0,0
Kp_KU6_circular246199_linear4868163,1,1
Kp_KU6_circular246199_linear4868163,1,1
Kp_KU12_circular68599_linear5202832,2,2
Kp_KU9_circular5569364_linear0,3,3
Kp_KU10_circular259723_linear4002906,3,3
Kp_KU8_circular248714_linear3980017,3,3
Kp_KU7_circular206344_linear5324540,4,4
Kp_KU13_circular5717070_linear0,4,4


In [2]:
import os
from paths import data_prod_path
from io_operations import presence_matrix
bk = 12
bn = 500
pk = 12
pn = 500
input_phage_path = f"encoded_sketches_data2/Phagemmh3_data2_n500_k12//"
input_bact_path = f"encoded_sketches_data2/Bactmmh3_data2_n500_k12//"
binary_matrix, entity_to_index, minhash_to_index, phage_minhash_data, bact_minhash_data = presence_matrix(
            phage_minhash_dir=os.path.join(data_prod_path, input_phage_path),
            bact_minhash_dir=os.path.join(data_prod_path, input_bact_path),
            k=[bk, pk], n=[bn, pn], reversecomp_data=False, TS=True, data2=True)

500 12 500 12
Loading phage minhash sketches from: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_data2/Phagemmh3_data2_n500_k12//
Loading bacteria minhash sketches from: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_data2/Bactmmh3_data2_n500_k12//
filepath: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_data2/Phagemmh3_data2_n500_k12//phage0_nepotimus_host_11.sig
filepath: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_data2/Phagemmh3_data2_n500_k12//phage1_arcadius_host_2.sig
filepath: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_data2/Phagemmh3_data2_n500_k12//phage2_pupienus_host_11.sig
filepath: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_data2/Phagemmh3_data2_n500_k12//phage3_rangabe_host_5.sig
filepath: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches_data2/Phagemmh3_d

In [3]:
phage_minhash_data

{'11': array([    70833702973837,   2723491219882070,   3771425091204336,
          3954405055067545,   5259727004609885,   6308387228922340,
          6401631391695504,   6589453547237979,   7641769630118091,
          8183000541723315,   8798907130895219,   9639084792224243,
          9771200895655798,  10468876691866861,  10510853357103892,
         10745930514243629,  11214464860971965,  11307275370358781,
         11640916667408772,  11906024349846020,  11995164843080552,
         12310317701662895,  13597980128395722,  13607682534235956,
         14021318027816199,  14112559972778837,  15723914283181215,
         16099023167686419,  16716197186276061,  18030967288081539,
         19070530580750032,  19431276110476242,  21693905228088605,
         22618265879954315,  22842908249626726,  22988352756497880,
         23175570292886874,  24499778600330233,  24841029058409802,
         25426694881582300,  25650131963245329,  26006735041829845,
         26213841667031296,  2672448528545

In [5]:
from io_operations import load_minhash_sketches
load_minhash_sketches(os.path.join(data_prod_path, input_phage_path))

{'Nepotimus_Host_11': [685938236316774,
  696271185614801,
  831465389594270,
  849760412837412,
  913574251591907,
  950261943178681,
  1177485889182630,
  1181641408704318,
  1207175381956643,
  1317079678039967,
  1541454507842219,
  1689695327408082,
  1782622072059304,
  1835513309109468,
  1887033482337261,
  1978612412727062,
  2082268969612586,
  2245076662607086,
  2622958663022711,
  2658682335133888,
  2666150476324738,
  2846447420325672,
  2854685073691783,
  3025736274491750,
  3139283695062194,
  3170493050760949,
  3199649202030076,
  3429721981632424,
  3575241721914454,
  3750190282418312,
  3817929232219282,
  3907679944992628,
  3921947100387200,
  3931229142664409,
  3953887642239603,
  3989025088804898,
  4026041238443051,
  4100238501509951,
  4126207050388173,
  4396638886696756,
  4450230535959668,
  4469879173127711,
  4610486339821830,
  4634069065173405,
  4660541218476132,
  4694931824847485,
  4738835171072153,
  4742354791045473,
  4754081068223290,
  483

In [1]:
from manipulations import clean_bact_names
K_names = ['Kp_KU7_circular_bp=206344_linear_bp=5324540', 'Kp_KU10_circular_bp=259723_linear_bp=4002906', 'Kp_KU1_circular_bp=5445670_linear_bp=0', 'Kp_KU11_circular_bp=5649217_linear_bp=0', 'Kp_KU6_circular_bp=246199_linear_bp=4868163']
H_names = ["Host 1", "Host 2", "Host 3", "Host 4", "Host 5", "Host 6", "Host 7", "Host 8", "Host 9", "Host 10", "Host 11", "Host 12"]

cleaned_K_names = clean_bact_names(K_names, data2=True)
print(cleaned_K_names)

['Host 7', 'Host 10', 'Host 1', 'Host 11', 'Host 6']
